# 05 · Formal evaluation, temporal validation and gate G4 — Neonatal mortality NCHS (M4)

| Field | Content |
|---|---|
| Framework sequence | 01: S1 · S2 · S3 → 02: S4 · G1 → 03: S5 · S6 · S7 · G2 · G3 → 04: S8 → **05: S8 · S9 · G4** → 06: S10 · S11 |
| What it does | Evaluates the primary learner (XGBoost), logistic regression and the reference model (gestational age + birthweight), runs the analyses of criteria C1 to C4 and classifies the model |
| Stages and gates | S8 (formal evaluation) · S9 (sensitivity and criteria) · G4 |
| Rules applied | C1 decided by the AUROC drop only (decision of 2026-11-23); M4-D1 (48 h), M4-D3 |
| Inputs | Contract table (notebook 03) and `data/outputs_04_selection.json` (notebook 04) |
| Outputs | `data/outputs_05_main_evaluation.txt`, figures in `figures/05_*.png`, local predictions in `data/predictions_05_*.parquet`, final model in `models/` |
| Time | 2 to 4 hours |
| Previous / next | `04_preliminary_learners` / `06_e10_action` |

In [ ]:
import sys, os, time, json
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy.optimize import brentq
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
# Language of the outputs: "es" in the Spanish notebook, "en" in the English one (the only line that differs).
LANG = "en"
import importlib
import libraries.i18n_es, libraries.i18n          # reloaded so that a running kernel picks up dictionary changes
importlib.reload(libraries.i18n_es); importlib.reload(libraries.i18n)
from libraries.i18n import translate
pd.set_option("display.max_colwidth", None)      # decision texts are shown in full
import warnings
# scikit-learn >= 1.8 announces that LogisticRegression(penalty=...) will be replaced by l1_ratio. The models keep
# penalty (the hyperparameters chosen in notebook 04) so that results stay exactly as reported; the notice is hidden.
warnings.filterwarnings("ignore", message=".*'penalty' was deprecated.*", category=FutureWarning)
lines_es = []
if LANG == "es":                       # progress messages (print) also in Spanish
    import builtins
    def print(*args, **kwargs):
        builtins.print(*[translate(a, "es") if isinstance(a, str) and not a.startswith("\n== ") else a for a in args], **kwargs)

def write_outputs(path, header):
    # English text output (cited by the documents) and, from the Spanish notebook, a Spanish copy (*_es.txt).
    path.write_text(header + "\n" + "\n".join(lines) + "\n", encoding="utf-8")
    if LANG == "es":
        path.with_name(path.stem + "_es.txt").write_text(translate(header, "es") + "\n" + "\n".join(lines_es) + "\n", encoding="utf-8")

CONTRACT_DIR = Path(os.environ.get("NCHS_CONTRACT_DIR", ROOT / "data" / "contract" / "neonatal_nchs_v1"))
OUT_FILE = ROOT / "data" / "outputs_05_main_evaluation.txt"
SELECTION_FILE = ROOT / "data" / "outputs_04_selection.json"
SEED = 20261123
DEV_YEARS, TEST_YEARS = list(range(2016, 2022)), [2022, 2023]
FRACTION = float(os.environ.get("NCHS_FRACTION", 0.02))
QUICK = os.environ.get("NCHS_QUICK") == "1"
N_JOBS = int(os.environ.get("NCHS_JOBS", -1))
REPEATS, FOLDS, B = (1, 3, 30) if QUICK else (3, 5, 200)
META = ["unit_id", "group_id", "period", "label_main", "at_risk_48h", "label_day2_27", "label_early_neonatal",
        "ge28", "ga_imputed", "bw_imputed", "n_unknown"]
PREDICTORS = [c for c in pq.read_schema(CONTRACT_DIR / "cohort_2021.parquet").names if c not in META]
REFERENCE = ["gestational_age", "birthweight"]           # reference model fixed in the outcome card
EXTRA = ["period", "label_main", "at_risk_48h", "label_day2_27", "label_early_neonatal", "ge28", "n_unknown"]

# Primary learner and hyperparameters chosen by the preliminary run (notebook 04, selection rule)
sel = json.loads(SELECTION_FILE.read_text(encoding="utf-8"))
assert QUICK or not sel["quick"], "notebook 04 was run in QUICK mode: run it again with the full sample"
primary, BEST = sel["primary"], sel["best"]

def make_model(name):
    cfg = dict(BEST[name]); n_iter = cfg.pop("n_estimators", None)
    imp = SimpleImputer(strategy="median")                       # decision M4-D3: no missingness indicators
    if name == "LR":
        return make_pipeline(imp, StandardScaler(), LogisticRegression(solver="liblinear", max_iter=500, **cfg))
    n_trees = 50 if QUICK else 300
    if name == "RF":
        return make_pipeline(imp, RandomForestClassifier(n_estimators=n_trees, n_jobs=N_JOBS, random_state=SEED, **cfg))
    if name == "ET":
        return make_pipeline(imp, ExtraTreesClassifier(n_estimators=n_trees, n_jobs=N_JOBS, random_state=SEED, **cfg))
    return make_pipeline(imp, XGBClassifier(n_estimators=(min(n_iter, 50) if QUICK else n_iter), tree_method="hist",
                                            n_jobs=N_JOBS, random_state=SEED, **cfg))

pd.set_option("display.max_colwidth", None)
lines = []
def show(title, obj):
    txt = obj.to_string() if hasattr(obj, "to_string") else str(obj)
    block = f"\n== {title}\n{txt}"
    print(block); lines.append(block)
SUPPRESS_SMALL = False
import contextlib, io, html as _html
_show_en = show

def _fmt(v):
    # Display format of a table cell: thousands separator for counts, 4 significant digits for small values.
    if isinstance(v, (bool, np.bool_)) or v is None or (isinstance(v, float) and np.isnan(v)):
        return v
    if SUPPRESS_SMALL and isinstance(v, (int, np.integer)) and 0 < v < 10:
        return "<10"
    if isinstance(v, (int, np.integer)) or (isinstance(v, (float, np.floating)) and float(v).is_integer() and abs(v) >= 1000):
        return f"{int(v):,}"
    if isinstance(v, (float, np.floating)):
        return f"{v:.4g}" if abs(v) < 1 else f"{v:,.4f}".rstrip("0").rstrip(".")
    return v

def _render(title, obj):
    # In Jupyter: a heading and an HTML table with scroll; elsewhere: the text block.
    try:
        from IPython import get_ipython
        from IPython.display import HTML, display as _display
        if get_ipython() is None:              # plain Python (not a notebook)
            raise ImportError
    except ImportError:
        return print(lines_es[-1] if LANG == "es" else lines[-1])
    head = f"<p style='margin:14px 0 4px 0'><b>{_html.escape(str(title))}</b></p>"
    if isinstance(obj, (pd.Series, pd.DataFrame)):
        if isinstance(obj, pd.Series):          # rows are different quantities: each cell on its own
            df = obj.astype(object).map(_fmt).to_frame(name=translate("value", LANG))
        else:                                    # columns: the same number of decimals down each column
            df = obj.copy().astype(object)
            for k in range(df.shape[1]):
                col = obj.iloc[:, k]
                if pd.api.types.is_float_dtype(col) and not col.dropna().map(float.is_integer).all():
                    dec = max((len(f"{v:.4g}".split(".")[1]) if "." in f"{v:.4g}" and "e" not in f"{v:.4g}" else 0)
                              for v in col.dropna()) if col.notna().any() else 0
                    df.iloc[:, k] = col.map(lambda v, d=dec: "" if pd.isna(v) else f"{v:,.{d}f}")
                else:
                    df.iloc[:, k] = col.map(_fmt)
        table = df.to_html(na_rep="", border=0)
        _display(HTML(head + f"<div style='max-height:420px; overflow:auto'>{table}</div>"))
    else:
        _display(HTML(head + f"<pre style='margin:0'>{_html.escape(str(obj))}</pre>"))

def show(title, obj):
    # The English block is kept in `lines` (data/outputs_*.txt) and, in Spanish, the translated block in
    # `lines_es` (data/outputs_*_es.txt); on screen the table is shown as an HTML table.
    with contextlib.redirect_stdout(io.StringIO()):
        _show_en(title, obj)
        if LANG == "es":
            _show_en(translate(title, "es"), translate(obj, "es"))
            lines_es.append(lines.pop())
    _render(translate(title, LANG), translate(obj, LANG))
show("Primary learner (notebook 04)", pd.Series({"primary learner": primary, "hyperparameters": json.dumps(BEST[primary]),
                                                  "ensemble adopted in notebook 04": sel["ensemble_adopted"]}))

## E8 · Modeling — development and temporal-test samples

Development 2016–2021 and temporal test 2022–2023, with all deaths and 2% of the other births, weighted.

In [ ]:
# Samples: every event and FRACTION of non-events, weight 1/FRACTION for non-events (the weighted
# sample stands for the full population). Development 2016-2021; temporal test 2022-2023.
def load(years, seed):
    rng = np.random.default_rng(seed); parts = []; n = e = 0
    for y in years:     # read in batches and keep the sample only, so the memory stays low
        for b in pq.ParquetFile(CONTRACT_DIR / f"cohort_{y}.parquet").iter_batches(500_000, columns=PREDICTORS + EXTRA):
            c = b.to_pandas()
            n += len(c); e += int(c["label_main"].sum())
            parts.append(c[c["label_main"].to_numpy() | (rng.random(len(c)) < FRACTION)])
    d = pd.concat(parts, ignore_index=True)
    if QUICK:
        d = d.sample(n=min(len(d), 40000), random_state=seed).reset_index(drop=True)
    d["w"] = np.where(d["label_main"], 1.0, 1.0 / FRACTION)
    return d, n, e

dev, n_dev, e_dev = load(DEV_YEARS, SEED)
test, n_test, e_test = load(TEST_YEARS, SEED + 1)
show("Data", pd.DataFrame({"births": [n_dev, n_test], "neonatal deaths": [e_dev, e_test],
                           "sample rows": [len(dev), len(test)]}, index=["development 2016-2021", "temporal test 2022-2023"]))

## E8 · C2 · Formal evaluation — recalibrated 3 × 5 cross-validation and comparison with the reference model

Every death occurs after the prediction point, so C2 is judged on the whole-population AUROC.

| Result | What is done | In this model |
|---|---|---|
| AUROC ≥ 0.75 and lower 95% limit ≥ 0.70 | C2 met | 0.964 (0.963–0.965): met |
| AUROC < 0.75 or lower limit < 0.70 | C2 not met: cannot be a predictor | — |
| Full model − reference | Shows how much is added beyond prematurity | +0.057 (0.056–0.058) |

In [ ]:
# Formal evaluation (framework S8-S9): repeated, recalibrated cross-validation of the primary learner,
# logistic regression and the reference model (the primary learner with gestational age and birthweight only).
# Metrics (weighted where the sampling matters; AUROC does not depend on it)
logit = lambda p: np.log(np.clip(p, 1e-7, 1 - 1e-7) / (1 - np.clip(p, 1e-7, 1 - 1e-7)))
expit = lambda z: 1 / (1 + np.exp(-z))

def cal_slope_intercept(y, p, w):
    lp = logit(p)
    slope = LogisticRegression(C=1e6, max_iter=1000).fit(lp.reshape(-1, 1), y, sample_weight=w).coef_[0, 0]
    f = lambda a: np.sum(w * (y - expit(a + lp)))        # calibration-in-the-large (offset = lp)
    return float(slope), float(brentq(f, -10, 10))

def metrics(y, p, w):
    o = np.argsort(p); cw = np.cumsum(w[o]); m = {}
    for share in (0.05, 0.10):
        cut = p[o][np.searchsorted(cw, (1 - share) * cw[-1])]
        m[f"events in top {int(share * 100)}% (%)"] = 100 * float(w[(p >= cut) & y].sum() / w[y].sum())
    slope, intercept = cal_slope_intercept(y, p, w)
    g = np.empty(len(p), int); g[o] = np.minimum((10 * cw / cw[-1]).astype(int), 9)   # weighted deciles
    ece = sum(abs(np.sum(w[g == k] * y[g == k]) - np.sum(w[g == k] * p[g == k])) for k in range(10)) / w.sum()
    m.update({"AUROC": roc_auc_score(y, p), "AUPRC": average_precision_score(y, p, sample_weight=w),
              "prevalence": float(np.sum(w * y) / w.sum()), "Brier": float(np.sum(w * (p - y) ** 2) / w.sum()),
              "calibration slope": slope, "calibration intercept": intercept,
              "observed/expected": float(np.sum(w * y) / np.sum(w * p)), "ECE": float(ece)})
    return m

def boot_ci(y, p, w, keys, seed):
    rng = np.random.default_rng(seed); pos, neg = np.where(y)[0], np.where(~y)[0]; res = []
    for _ in range(B):
        ix = np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))])
        res.append({k: v for k, v in metrics(y[ix], p[ix], w[ix]).items() if k in keys})
    r = pd.DataFrame(res)
    return r.quantile(0.025), r.quantile(0.975)

def fit_calibrated(name, X, y, w, seed, cols=None):
    """Fit on 80% of the training part and Platt-recalibrate on the other 20% (inside training)."""
    cols = cols or PREDICTORS
    X = X[cols]
    X_f, X_c, y_f, y_c, w_f, w_c = train_test_split(X, y, w, test_size=0.2, stratify=y, random_state=seed)
    m = make_model(name); step = m.steps[-1][0]
    m.fit(X_f, y_f, **{f"{step}__sample_weight": w_f})
    platt = LogisticRegression(C=1e6, max_iter=1000).fit(logit(m.predict_proba(X_c)[:, 1]).reshape(-1, 1), y_c, sample_weight=w_c)
    predict = lambda Xn: platt.predict_proba(logit(m.predict_proba(Xn[cols])[:, 1]).reshape(-1, 1))[:, 1]
    predict.model, predict.platt, predict.cols = m, platt, cols      # kept so the final model can be saved (models/)
    return predict

# Repeated stratified cross-validation (births are independent units; no mother identifier):
# REPEATS x FOLDS, recalibration inside each training part, out-of-fold predictions pooled per repeat.
X_dev, y_dev, w_dev = dev[PREDICTORS], dev["label_main"].to_numpy(bool), dev["w"].to_numpy()
MODELS = {primary: (primary, PREDICTORS), "LR": ("LR", PREDICTORS), "reference (GA + birthweight)": (primary, REFERENCE)}
if primary == "LR":
    MODELS = {"LR": ("LR", PREDICTORS), "reference (GA + birthweight)": ("LR", REFERENCE)}
oof = {name: np.zeros((REPEATS, len(dev))) for name in MODELS}
for r in range(REPEATS):
    t = time.time()
    for tr, te in StratifiedKFold(FOLDS, shuffle=True, random_state=SEED + r).split(X_dev, y_dev):
        for name, (learner, cols) in MODELS.items():
            f = fit_calibrated(learner, X_dev.iloc[tr], y_dev[tr], w_dev[tr], SEED + r, cols)
            oof[name][r, te] = f(X_dev.iloc[te])
    print(f"repeat {r + 1} done in {(time.time() - t) / 60:.1f} min")

KEYS = ["AUROC", "AUPRC", "calibration slope", "calibration intercept", "observed/expected"]
rows = {}
for name in oof:
    per_rep = pd.DataFrame([metrics(y_dev, oof[name][r], w_dev) for r in range(REPEATS)])
    lo, hi = boot_ci(y_dev, oof[name][0], w_dev, KEYS, SEED)
    row = per_rep.mean().round(4).to_dict()
    row.update({f"{k} 2.5%": round(lo[k], 4) for k in KEYS}); row.update({f"{k} 97.5%": round(hi[k], 4) for k in KEYS})
    row["AUROC SD across repeats"] = round(per_rep["AUROC"].std(ddof=0), 4)
    rows[name] = row
main = pd.DataFrame(rows)
show(f"Internal validation: {REPEATS} x {FOLDS}-fold cross-validation, recalibrated (development 2016-2021)", main)

# What the rest of the information adds to gestational age and birthweight (paired bootstrap, repeat 1)
rng = np.random.default_rng(SEED + 5); pos, neg = np.where(y_dev)[0], np.where(~y_dev)[0]
REF = "reference (GA + birthweight)"
d_auc = []
for _ in range(B):
    ix = np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))])
    d_auc.append(roc_auc_score(y_dev[ix], oof[primary][0][ix]) - roc_auc_score(y_dev[ix], oof[REF][0][ix]))
show("Full model minus reference model (GA + birthweight): AUROC difference, paired bootstrap 95% CI",
     f"{roc_auc_score(y_dev, oof[primary][0]) - roc_auc_score(y_dev, oof[REF][0]):+.4f} "
     f"({np.percentile(d_auc, 2.5):+.4f} to {np.percentile(d_auc, 97.5):+.4f})")

## E9 · C4 · Stability over time — temporal validation 2022–2023

Fit once on 2016–2021, recalibration inside it and a single test on 2022–2023.

| Result | What is done | In this model |
|---|---|---|
| Temporal AUROC ≥ 0.70 and slope between 0.90 and 1.10 | C4 met | 0.961; slope 1.01: met |
| Slope outside the band | C4 not met: calibration cannot be fixed with the intercept alone | — |
| Temporal AUROC < 0.65 | Insufficient signal, regardless of the rest | — |

In [ ]:
# Temporal validation (criterion C4): fit once on the whole development sample (2016-2021, which passes
# gate G3, notebook 03), recalibrate inside it, apply once to 2022-2023.
X_t, y_t, w_t = test[PREDICTORS], test["label_main"].to_numpy(bool), test["w"].to_numpy()
temporal, p_temporal = {}, {}
FINAL = {}                                   # fitted models, kept to save the final one (models/)
for name, (learner, cols) in MODELS.items():
    f = fit_calibrated(learner, X_dev, y_dev, w_dev, SEED, cols)
    p_t = p_temporal[name] = f(X_t)
    FINAL[name] = f
    m = metrics(y_t, p_t, w_t); lo, hi = boot_ci(y_t, p_t, w_t, KEYS, SEED + 7)
    temporal[name] = {**{k: round(v, 4) for k, v in m.items()},
                      **{f"{k} 2.5%": round(lo[k], 4) for k in KEYS}, **{f"{k} 97.5%": round(hi[k], 4) for k in KEYS}}
show("Temporal validation: development 2016-2021, single test 2022-2023", pd.DataFrame(temporal))

## E9 · C1 · Events already present — 48-hour landmark on the model fitted once

Out-of-fold predictions are restricted to newborns alive at the start of day 2 (M4-D1), where the event is death on days 2 to 27.

| Result | What is done | In this model |
|---|---|---|
| AUROC drop ≤ 0.05 | C1 met | 0.032 (0.031–0.033): met |
| Drop > 0.05 | C1 not met: the model depends on cases already under way | — |
| AUPRC gain kept | Reported; does not decide | 26% |

In [ ]:
# Criterion C1 (framework 1.2): evaluated on the model fitted once on the whole population. Its out-of-fold
# predictions are restricted to the newborns alive at the start of day 2 (48-hour landmark, decision M4-D1),
# where the event is death on days 2-27. Paired bootstrap of the full population and the landmark subset.
p0 = oof[primary][0]
lm = dev["at_risk_48h"].to_numpy(bool); y_lm = dev["label_day2_27"].to_numpy(bool)
def c1_quantities(ix):
    yf, pf, wf = y_dev[ix], p0[ix], w_dev[ix]
    sub = lm[ix]; ys, ps, ws = y_lm[ix][sub], pf[sub], wf[sub]
    a_f, a_s = roc_auc_score(yf, pf), roc_auc_score(ys, ps)
    prev_f, prev_s = np.sum(wf * yf) / wf.sum(), np.sum(ws * ys) / ws.sum()
    g_f = average_precision_score(yf, pf, sample_weight=wf) - prev_f
    g_s = average_precision_score(ys, ps, sample_weight=ws) - prev_s
    return {"AUROC full": a_f, "AUROC landmark": a_s, "AUROC drop": a_f - a_s,
            "AUPRC gain full": g_f, "AUPRC gain landmark": g_s, "AUPRC gain kept (%)": 100 * g_s / g_f}
point = c1_quantities(np.arange(len(dev)))
rng = np.random.default_rng(SEED + 11); pos, neg = np.where(y_dev)[0], np.where(~y_dev)[0]
bs = pd.DataFrame([c1_quantities(np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))])) for _ in range(B)])
c1_tab = pd.DataFrame({"estimate": pd.Series(point), "2.5%": bs.quantile(0.025), "97.5%": bs.quantile(0.975)}).round(4)
show(f"Criterion C1: 48-hour landmark on the out-of-fold predictions of the primary learner "
     f"({int(y_lm[lm].sum())} deaths on days 2-27; {int((y_dev & ~lm).sum())} deaths on days 0-1 removed)", c1_tab)

## E9 · Sensitivity and robustness — analyses of the outcome card on the model fitted once

28 weeks or more, 48 h at 28 weeks or more, early neonatal, pandemic and complete records, restricting the same predictions.

| Result | What is done | In this model |
|---|---|---|
| AUROC in the subgroup ≥ 0.75 | The result does not depend on that subgroup | ≥ 28 weeks 0.936; complete records 0.959 |
| O/E far from 1 in a period | Local recalibration before using the model | 2020–2021: 0,94 |

In [ ]:
# Sensitivity analyses of the outcome card (S9) on the same out-of-fold predictions (no refit)
ge28 = dev["ge28"].to_numpy(bool); early = dev["label_early_neonatal"].to_numpy(bool)
late = y_dev & ~early; pandemic = dev["period"].isin([2020, 2021]).to_numpy()
complete = (dev["n_unknown"] == 0).to_numpy()
sens = {"all births (main)": (np.ones(len(dev), bool), y_dev),
        "(c) 28 weeks or more": (ge28, y_dev),
        "(a) + (c) 48-hour landmark, 28 weeks or more": (lm & ge28, y_lm),
        "(b) early neonatal death 0-6 days vs survivors (deaths on days 7-27 removed)": (~late, early),
        "cohorts 2020-2021 (pandemic)": (pandemic, y_dev),
        "cohorts 2016-2019": (~pandemic, y_dev),
        "complete records (main model, no refit)": (complete, y_dev)}
srows = {}
for k, (mask, yy) in sens.items():
    ww, pp = w_dev[mask], p0[mask]
    srows[k] = {"births (weighted)": int(round(ww.sum())), "events": int(yy[mask].sum()),
                "AUROC": round(roc_auc_score(yy[mask], pp), 4),
                "observed/expected": round(float(np.sum(ww * yy[mask]) / np.sum(ww * pp)), 3)}
show("Sensitivity analyses (primary learner, out-of-fold predictions of the model fitted once; observed/expected "
     "compares the deaths of each label with the predicted risk of death at 0-27 days)", pd.DataFrame(srows).T)

## E9 · Sensitivity and robustness — refitted analyses

Same learner and hyperparameters, fitted inside each subset.

| Result | What is done | In this model |
|---|---|---|
| Refitted ≈ not refitted | The result does not depend on the imputation or the population | Complete records 0.960 vs 0.959 |
| Full model ≫ reference at ≥ 28 weeks | The model adds information beyond prematurity | 0.936 vs 0.829 |

In [ ]:
# Refitted sensitivities (each cross-validated with the primary learner and its tuned hyperparameters).
# Complete records (decision M4-D3): the set has tens of thousands of events, far above the 104 required by
# gate G3 for 71 parameters (notebook 03). Predictions are kept for the figures.
REFIT_P = {}
def cv_auroc(name, mask, cols, label):
    Xs, ys, ws = dev.loc[mask, cols], dev.loc[mask, label].to_numpy(bool), w_dev[mask]
    pp = np.zeros(len(Xs))
    for tr, te in StratifiedKFold(FOLDS, shuffle=True, random_state=SEED).split(Xs, ys):
        pp[te] = fit_calibrated(primary, Xs.iloc[tr], ys[tr], ws[tr], SEED, cols)(Xs.iloc[te])
    REFIT_P[name] = (ys, pp)
    return round(roc_auc_score(ys, pp), 4), int(ys.sum())
rrows = {n: cv_auroc(n, m, c, l) for n, m, c, l in [
    ("complete records, refitted (no imputation)", complete, PREDICTORS, "label_main"),
    ("48-hour landmark, refitted", lm, PREDICTORS, "label_day2_27"),
    ("28 weeks or more, refitted", ge28, PREDICTORS, "label_main"),
    ("28 weeks or more, reference model (GA + birthweight), refitted", ge28, REFERENCE, "label_main")]}
show("Refitted sensitivity analyses (primary learner): AUROC and events", pd.DataFrame(rrows, index=["AUROC", "events"]).T)

## P4 · Classification gate — predictor, early identifier or insufficient signal

Rule of the framework; C1 is decided by the AUROC drop (`docs/decision_G4_C1_2026-11-23.md`). The output of this run still shows a row «classification under framework 1.2», left by the code; it is not a classification of the model.

| Result | What is done | In this model |
|---|---|---|
| AUROC ≥ 0.75 and meets C1, C2 and C4 (C3 not applicable) | **Predictor**: prevention strategy (notebook 06) | **Predictor** |
| AUROC ≥ 0.75 and top 10% ≥ 2× chance, but fails C1, C3 or C4 | **Early identifier**: surveillance and prioritization; returns are considered | — |
| Otherwise, or temporal AUROC < 0.65 | **Insufficient signal**: reported | — |

In [ ]:
# Gate G4 (classification, framework 1.3 of 2026-11-23; see docs/decision_G4_C1_2026-11-23.md).
# C1: 48-hour landmark on the model fitted once: AUROC drop <= 0.05. The share of the AUPRC gain kept is still
# reported, and the decision of framework 1.2 (which also required >= 2/3 of it) is shown for traceability.
# C2: every death occurs after the prediction point (birth + 5 minutes): AUROC >= 0.75 with lower 95% limit >= 0.70.
# C3: not applicable (the outcome comes from the death certificate, not from a test).
# C4: temporal AUROC >= 0.70 and temporal calibration slope between 0.90 and 1.10.
x, tt = main[primary], temporal[primary]
c1 = point["AUROC drop"] <= 0.05                                                    # framework 1.3
c1_v12 = c1 and (point["AUPRC gain kept (%)"] >= 100 * 2 / 3)                       # framework 1.2, for traceability
c2 = (x["AUROC"] >= 0.75) and (x["AUROC 2.5%"] >= 0.70)
c4 = (tt["AUROC"] >= 0.70) and (0.90 <= tt["calibration slope"] <= 1.10)
top10 = x["events in top 10% (%)"] / 10
def classify(c1_met):
    if tt["AUROC"] < 0.65:
        return "NEITHER"
    if x["AUROC"] >= 0.75 and c1_met and c2 and c4:
        return "PREDICTOR"
    return "EARLY IDENTIFIER" if (x["AUROC"] >= 0.75 and top10 >= 2) else "NEITHER"
cls = classify(c1)
show("Gate G4: classification of the neonatal model (primary learner)", pd.Series({
    "C1 events already present": f"{'met' if c1 else 'NOT met'} (48-hour landmark: AUROC drop {point['AUROC drop']:.3f}, "
        f"95% CI {c1_tab.loc['AUROC drop', '2.5%']:.3f}-{c1_tab.loc['AUROC drop', '97.5%']:.3f}, required <= 0.05; "
        f"AUPRC gain kept {point['AUPRC gain kept (%)']:.0f}%, 95% CI {c1_tab.loc['AUPRC gain kept (%)', '2.5%']:.0f}-"
        f"{c1_tab.loc['AUPRC gain kept (%)', '97.5%']:.0f}%, reported, not decisive in framework 1.3; "
        f"framework 1.2 required >= 67%: {'met' if c1_v12 else 'NOT met'})",
    "C2 outcome with time": f"{'met' if c2 else 'NOT met'} (AUROC {x['AUROC']:.3f}, 95% CI {x['AUROC 2.5%']:.3f}-{x['AUROC 97.5%']:.3f}; required >= 0.75 and >= 0.70)",
    "C3 detection bias": "not applicable (death certificate; linkage of deaths to births 100%, notebook 01)",
    "C4 stability over time": f"{'met' if c4 else 'NOT met'} (temporal AUROC {tt['AUROC']:.3f}, 95% CI {tt['AUROC 2.5%']:.3f}-{tt['AUROC 97.5%']:.3f}; "
        f"slope {tt['calibration slope']:.2f}, 95% CI {tt['calibration slope 2.5%']:.2f}-{tt['calibration slope 97.5%']:.2f}, required 0.90-1.10)",
    "events in top 10% vs chance": f"{top10:.2f} times",
    "classification": cls,
    "classification under framework 1.2 (traceability)": classify(c1_v12)}))
# Accuracy and confusion matrix of the primary model (requested; not a framework metric)
cm = []
for share in (0.10, 0.05):
    p = oof[primary][0]; o = np.argsort(p); cw = np.cumsum(w_dev[o])
    flag = p >= p[o][np.searchsorted(cw, (1 - share) * cw[-1])]
    tp, fn = w_dev[flag & y_dev].sum(), w_dev[~flag & y_dev].sum()
    fp, tn = w_dev[flag & ~y_dev].sum(), w_dev[~flag & ~y_dev].sum()
    cm.append({"flagged": f"top {int(share * 100)}%", "TP": round(tp), "FN": round(fn), "FP": round(fp), "TN": round(tn),
               "accuracy": round((tp + tn) / (tp + tn + fp + fn), 4), "sensitivity": round(tp / (tp + fn), 3),
               "specificity": round(tn / (tn + fp), 4), "PPV": round(tp / (tp + fp), 4), "NPV": round(tn / (tn + fn), 5)})
show("Confusion matrix of the primary learner (out-of-fold, weighted to the development population)", pd.DataFrame(cm).set_index("flagged"))

# Decision curve (net benefit) at a few risk thresholds
dc = []
for thr in (0.002, 0.005, 0.01, 0.02, 0.05):
    p = oof[primary][0]; flag = p >= thr; n = w_dev.sum()
    nb = w_dev[flag & y_dev].sum() / n - w_dev[flag & ~y_dev].sum() / n * thr / (1 - thr)
    prev = w_dev[y_dev].sum() / n
    dc.append({"threshold": thr, "net benefit model": round(nb, 5), "net benefit treat all": round(prev - (1 - prev) * thr / (1 - thr), 5),
               "% flagged": round(100 * w_dev[flag].sum() / n, 2)})
show("Decision curve (primary learner)", pd.DataFrame(dc).set_index("threshold"))

## Figures · ROC curves and confusion matrices

ROC curves of the final model, logistic regression and the reference model (internal and temporal), and of the sensitivity analyses (not refitted and refitted). Confusion matrices weighted to the population at the top-10% risk cut. Accuracy and F1 are shown but do not decide in the framework: with a prevalence of 0.28%, flagging nobody already gives an accuracy of 0.997.

In [ ]:
# Figure helpers (shared by the figure and summary cells). Curves use the out-of-fold or test predictions;
# confusion matrices are weighted to the full population (non-events carry weight 1/FRACTION).
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
FIG_DIR = ROOT / "figures"
FIG_DIR.mkdir(exist_ok=True)

def plot_roc(curves, title, fname):
    """curves: list of (label, y, p). One subplot per curve (algorithm or population), so the curves do not overlap.
    AUROC does not depend on the case-control weights."""
    n = len(curves); ncol = min(3, n); nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(4.4 * ncol, 4.4 * nrow), squeeze=False)
    for k, (label, yy, pp) in enumerate(curves):
        ax = axes[k // ncol][k % ncol]
        fpr, tpr, _ = roc_curve(yy, pp)
        ax.plot(fpr, tpr, lw=1.8, color="#2F6DB5")
        ax.fill_between(fpr, tpr, alpha=0.08, color="#2F6DB5")
        ax.plot([0, 1], [0, 1], ls="--", color="grey", lw=1)
        ax.set_title(f"{translate(label, LANG)}\nAUROC {roc_auc_score(yy, pp):.3f}", fontsize=10)
        ax.set_xlim(0, 1); ax.set_ylim(0, 1.01); ax.set_aspect("equal")
        ax.set_xlabel(translate("1 - specificity (false positive rate)", LANG), fontsize=9)
        ax.set_ylabel(translate("Sensitivity (true positive rate)", LANG), fontsize=9)
    for k in range(n, nrow * ncol):
        axes[k // ncol][k % ncol].axis("off")
    fig.suptitle(translate(title, LANG), fontsize=12)
    fig.tight_layout(); fig.savefig(FIG_DIR / (fname.replace(".png", "_es.png") if LANG == "es" else fname), dpi=200); plt.show()

def cut_top(p, w, share):
    """Risk cut that flags the given share of the weighted population."""
    o = np.argsort(-p); cw = np.cumsum(w[o])
    return float(p[o][np.searchsorted(cw, share * cw[-1])])

def threshold_metrics(y, p, w, cut):
    """Accuracy, F1 and the confusion-matrix rates at a declared risk cut (weighted to the population)."""
    flag = p >= cut
    tp, fp = w[flag & y].sum(), w[flag & ~y].sum(); fn, tn = w[~flag & y].sum(), w[~flag & ~y].sum()
    sens, ppv = tp / (tp + fn), tp / (tp + fp)
    return {"risk cut": round(cut, 4), "TP": int(round(tp)), "FP": int(round(fp)), "FN": int(round(fn)), "TN": int(round(tn)),
            "accuracy": round(float((tp + tn) / (tp + fp + fn + tn)), 4), "F1": round(float(2 * sens * ppv / (sens + ppv)), 4),
            "sensitivity": round(float(sens), 4), "specificity": round(float(tn / (tn + fp)), 4),
            "PPV": round(float(ppv), 4), "NPV": round(float(tn / (tn + fn)), 5)}

def plot_cm(panels, fname):
    """panels: list of (title, y, p, w, cut). For each panel, on the left the classification report of the two classes
    (precision, recall, F1, accuracy and macro average) as a heatmap and, on the right, the confusion matrix. Counts are
    weighted to the full population (non-events carry weight 1/FRACTION); class 1 = neonatal death."""
    from sklearn.metrics import ConfusionMatrixDisplay
    fig, axes = plt.subplots(len(panels), 2, figsize=(12.5, 4.6 * len(panels)), squeeze=False,
                             gridspec_kw={"width_ratios": [1.35, 1]})
    for row, (title, yy, pp, ww, cut) in enumerate(panels):
        m = threshold_metrics(yy, pp, ww, cut)
        tp, fp, fn, tn = m["TP"], m["FP"], m["FN"], m["TN"]
        f1 = lambda a, b: 2 * a * b / (a + b) if (a + b) else 0.0
        p0, r0 = tn / (tn + fn), tn / (tn + fp)
        p1, r1 = tp / (tp + fp), tp / (tp + fn)
        acc = (tp + tn) / (tp + fp + fn + tn)
        rep = np.array([[p0, r0, f1(p0, r0)], [p1, r1, f1(p1, r1)], [acc, acc, acc],
                        [(p0 + p1) / 2, (r0 + r1) / 2, (f1(p0, r0) + f1(p1, r1)) / 2]])
        ax = axes[row][0]
        ax.imshow(rep, cmap="YlGnBu", vmin=0, vmax=1, aspect="auto")
        for i in range(rep.shape[0]):
            for j in range(rep.shape[1]):
                ax.text(j, i, f"{rep[i, j]:.2f}", ha="center", va="center", fontsize=10,
                        color="white" if rep[i, j] > 0.6 else "black")
        ax.set_xticks(range(3), [translate(c, LANG) for c in ("precision", "recall", "f1-score")])
        ax.set_yticks(range(4), ["0", "1", translate("accuracy", LANG), translate("macro avg", LANG)])
        ax.set_xlabel(translate("Metrics", LANG)); ax.set_ylabel(translate("Classes", LANG))
        ax.set_title(translate(title, LANG) + f" ({translate('risk cut', LANG)} {cut:.4f})", fontsize=9)
        ax2 = axes[row][1]
        ConfusionMatrixDisplay(confusion_matrix=np.array([[tn, fp], [fn, tp]]), display_labels=[0, 1]).plot(
            ax=ax2, cmap="Blues", colorbar=False, values_format=",d")
        ax2.set_xlabel(translate("Predicted label", LANG)); ax2.set_ylabel(translate("True label", LANG))
        ax2.set_title(translate("Confusion matrix", LANG), fontsize=9)
    fig.tight_layout(); fig.savefig(FIG_DIR / (fname.replace(".png", "_es.png") if LANG == "es" else fname), dpi=200); plt.show()

plot_roc([(k, y_dev, oof[k][0]) for k in MODELS], "Internal validation 2016-2021 (out-of-fold, repeat 1)", "05_roc_internal.png")
plot_roc([(k, y_t, p_temporal[k]) for k in MODELS], "Temporal validation 2022-2023", "05_roc_temporal.png")
plot_roc([("all births (main)", y_dev, p0), ("28 weeks or more", y_dev[ge28], p0[ge28]),
          ("48-hour landmark (deaths on days 2-27)", y_lm[lm], p0[lm]),
          ("early neonatal 0-6 days", early[~late], p0[~late]), ("complete records", y_dev[complete], p0[complete])],
         "Sensitivity analyses: model fitted once, predictions restricted", "05_roc_sensitivity.png")
plot_roc([(k, ys, pp) for k, (ys, pp) in REFIT_P.items()], "Refitted sensitivity analyses", "05_roc_refitted.png")
plot_cm([("Internal 2016-2021, top 10% of risk", y_dev, p0, w_dev, cut_top(p0, w_dev, 0.10)),
         ("Temporal 2022-2023, top 10% of risk", y_t, p_temporal[primary], w_t, cut_top(p_temporal[primary], w_t, 0.10))],
        "05_confusion_matrices.png")
# Predictions kept locally for later figures (data/ is not versioned; no identifiers)
pd.DataFrame({"set": "development", "label": y_dev, "w": w_dev, **{f"p_{k}": oof[k][0] for k in MODELS}}).to_parquet(ROOT / "data" / "predictions_05_development.parquet", index=False)
pd.DataFrame({"set": "temporal", "label": y_t, "w": w_t, **{f"p_{k}": p_temporal[k] for k in MODELS}}).to_parquet(ROOT / "data" / "predictions_05_temporal.parquet", index=False)

## E11 · Final model saved for reuse

The model fitted once on 2016–2021 with its Platt recalibration (the one validated on 2022–2023) is saved in `models/`, so it can be applied to new births without the data; `libraries/neonatal_model.py` shows how. TRIPOD+AI items 12g and 22.

| Result | What is done | In this model |
|---|---|---|
| Reloaded and portable files reproduce the predictions (difference ≈ 0) | The files are published with the code | On running 05 |
| Difference | Check the library versions against `requirements.txt` | — |

In [ ]:
# Final model saved for reuse (TRIPOD+AI items 12g and 22): the primary learner fitted once on the whole development
# sample 2016-2021 with its Platt recalibration, i.e. the model evaluated on 2022-2023 (criterion C4). Files in models/:
# the full pipeline (joblib; needs the versions of requirements.txt), the learner in a portable format (XGBoost JSON)
# and a card with the predictor order, the imputation medians and the Platt coefficients. No data are stored.
# Use: libraries/neonatal_model.py. The check below reloads the files and predicts the temporal test again.
import joblib, platform, sklearn, xgboost
from libraries.neonatal_model import load_model, predict_risk, predict_from_card
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)
fm = FINAL[primary]
F_JOBLIB, F_CARD, F_XGB = (MODEL_DIR / "neonatal_final_model.joblib", MODEL_DIR / "neonatal_final_model_card.json",
                           MODEL_DIR / "neonatal_final_xgb.json")
joblib.dump({"pipeline": fm.model, "platt": fm.platt, "predictors": list(fm.cols), "learner": primary}, F_JOBLIB)
card = {"model": "M4 neonatal death at 0-27 days, NCHS linked birth/infant death files, live births >= 22 weeks",
        "prediction_point": "birth + 5 minutes", "learner": primary, "hyperparameters": BEST[primary],
        "development_cohorts": DEV_YEARS, "temporal_test_cohorts": TEST_YEARS,
        "sampling": f"all deaths and {FRACTION:.0%} of the other births; non-deaths weighted 1/{FRACTION}",
        "recalibration": "Platt on 20% of the development sample: risk = 1 / (1 + exp(-(intercept + slope_on_logit * logit(p))))",
        "seed": SEED, "quick_mode": QUICK, "predictors": list(fm.cols),
        "imputation_medians": {c: float(v) for c, v in zip(fm.cols, fm.model[0].statistics_)},
        "platt": {"intercept": float(fm.platt.intercept_[0]), "slope_on_logit": float(fm.platt.coef_[0, 0])},
        "versions": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
                     "scikit-learn": sklearn.__version__, "xgboost": xgboost.__version__, "joblib": joblib.__version__}}
F_CARD.write_text(json.dumps(card, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
if primary == "XGB":
    fm.model[-1].get_booster().save_model(str(F_XGB))
# Check: the reloaded files give the same predictions as the model in memory (temporal test 2022-2023)
p_reload = predict_risk(load_model(F_JOBLIB), test)
p_card = predict_from_card(F_CARD, F_XGB, test) if primary == "XGB" else None
size = lambda f: f"{f.relative_to(ROOT).as_posix()} ({f.stat().st_size / 1024:,.0f} KB)"
show("Final model saved for reuse (models/)", pd.Series({
    "learner": primary,
    "pipeline with Platt recalibration (joblib)": size(F_JOBLIB),
    "learner in portable format (XGBoost JSON)": size(F_XGB) if primary == "XGB" else "not saved (the primary learner is not XGBoost)",
    "model card (predictors, imputation medians, Platt coefficients)": size(F_CARD),
    "check: max |difference|, reloaded joblib vs model in memory": float(np.max(np.abs(p_reload - p_temporal[primary]))),
    "check: max |difference|, portable files vs model in memory":
        float(np.max(np.abs(p_card - p_temporal[primary]))) if p_card is not None else "not applicable",
    "temporal AUROC with the reloaded model": round(roc_auc_score(y_t, p_reload), 4)}))
assert np.allclose(p_reload, p_temporal[primary], atol=1e-9), "the reloaded model does not reproduce the predictions"
if p_card is not None:
    assert np.allclose(p_card, p_temporal[primary], atol=1e-5), "the portable files do not reproduce the predictions"


## Notebook summary

Discrimination, calibration and threshold metrics (accuracy, F1, sensitivity, specificity, PPV, NPV) by model and set, and the result of gate G4.

In [ ]:
# Summary of notebook 05: one row per model and validation set. Threshold metrics (accuracy, F1, sensitivity,
# specificity, PPV, NPV) at the top-10% risk cut of each set; they are reported, the framework does not decide with them.
def ci(src, k):
    return f"{src[k]:.3f} ({src[k + ' 2.5%']:.3f}-{src[k + ' 97.5%']:.3f})"
rows_s = {}
for name in MODELS:
    for setname, (yy, pp, ww, src) in {"internal 2016-2021": (y_dev, oof[name][0], w_dev, main[name]),
                                       "temporal 2022-2023": (y_t, p_temporal[name], w_t, temporal[name])}.items():
        tm = threshold_metrics(yy, pp, ww, cut_top(pp, ww, 0.10))
        rows_s[f"{name} · {setname}"] = {"AUROC (95% CI)": ci(src, "AUROC"), "AUPRC": round(src["AUPRC"], 3),
            "Brier": round(src["Brier"], 5), "calibration slope": round(src["calibration slope"], 3),
            "observed/expected": round(src["observed/expected"], 3), "events in top 10% (%)": round(src["events in top 10% (%)"], 1),
            **{k: tm[k] for k in ["accuracy", "F1", "sensitivity", "specificity", "PPV", "NPV"]}}
summary = pd.DataFrame(rows_s).T
show("Summary: discrimination, calibration and threshold metrics (top-10% cut)", summary)
show("Summary: gate G4", pd.Series({
    "C1 (48-hour landmark): AUROC drop": f"{point['AUROC drop']:.3f} ({c1_tab.loc['AUROC drop', '2.5%']:.3f}-{c1_tab.loc['AUROC drop', '97.5%']:.3f})",
    "C1 met (framework 1.3)": bool(c1), "C2 met": bool(c2), "C3": "not applicable", "C4 met": bool(c4),
    "classification (framework 1.3)": cls, "classification under framework 1.2": classify(c1_v12)}))
write_outputs(OUT_FILE, "Main evaluation of the neonatal model (birth + 5 minutes)")
translate(summary, LANG)